### Chatbot using LangGraph

In [28]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langgraph.graph.message import BaseMessage, add_messages
from langgraph.checkpoint.memory import MemorySaver

In [29]:
load_dotenv()

True

In [30]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.6, max_tokens=500)

In [31]:
# Define state for chatbot
class ChatbotState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [32]:
# Define state graph
graph = StateGraph(ChatbotState)

In [33]:
# Nodes
def chat_node(state: ChatbotState):
    # user input
    messages = state['messages']

    # llm response
    response = model.invoke(messages)

    # add into messages
    return {"messages": [response]}


In [34]:
# Define nodes
graph.add_node('chat_node', chat_node)

# Define edges
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

In [35]:
memory = MemorySaver()    # Save in RAM
chatbot = graph.compile(checkpointer=memory)

In [36]:
from langchain_core.messages import HumanMessage

thread_id = "chat_thread_1"  # Unique identifier for the chat thread

while True:
    user_input = input("Type message: ")

    print(f"User: {user_input}")

    if user_input.strip().lower() in ["exit", "quit", "bye"]:
        break

    # configuration for the memory saver to store messages in the same thread    
    config = {"configurable": {"thread_id": thread_id}}

    # Create initial state with user input
    initial_state = {"messages": [HumanMessage(content=user_input)]}

    # Invoke the chatbot workflow
    result = chatbot.invoke(initial_state, config=config)

    # Get the response from the result
    response_message = result['messages'][-1].content
    print(f"Chatbot: {response_message}")

User: Hello Bot
Chatbot: Hello! How can I assist you today?
User: My name is Vishal and can you help me to write 1 sentence about me
Chatbot: Of course, Vishal! Could you provide me with a few details about yourself, such as your interests, profession, or anything specific you'd like to highlight?
User: add line about AI proffesinal
Chatbot: Sure! Here’s a sentence for you: 

"Vishal is an AI professional passionate about leveraging technology to drive innovation and solve complex problems." 

Feel free to modify it or let me know if you'd like something different!
User: update line and say full stack
Chatbot: Certainly! Here’s the updated sentence: 

"Vishal is a full-stack AI professional passionate about leveraging technology to drive innovation and solve complex problems." 

Let me know if you need any further adjustments!
User: No thanks
Chatbot: You're welcome! If you have any more questions or need assistance with anything else, feel free to ask. Have a great day!
User: Do you k